# Oracle Service Request JSON RAG Ingestion
Reads all SR JSON files from `./sr_json` and creates only `./vectordb_sr` with collection `service_requests`.


In [ ]:
# CELL 1: Configuration
from pathlib import Path
SR_JSON_DIR=Path('./sr_json')
CHROMA_DIR=Path('./vectordb_sr')
COLLECTION_NAME='service_requests'
EMBEDDING_MODEL='BAAI/bge-small-en-v1.5'
CHUNK_SIZE=1200
CHUNK_OVERLAP=200
print('JSON dir:', SR_JSON_DIR.resolve())


In [ ]:
# CELL 2: Imports and helpers
import json,re,shutil
from typing import Any
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

def clean_text(value: Any)->str:
    if value is None: return ''
    text=str(value).replace('\x00',' ')
    text=re.sub(r'[ \t]+',' ',text)
    text=re.sub(r'\n{3,}','\n\n',text)
    return text.strip()

def build_document(data, path):
    sr_id=clean_text(data.get('sr_id') or data.get('sr_number'))
    title=clean_text(data.get('title') or data.get('subject'))
    subject=clean_text(data.get('subject'))
    description=clean_text(data.get('description'))
    history=clean_text(data.get('text'))
    source=clean_text(data.get('sr_details_url') or data.get('source_url') or str(path))
    parts=[]
    if sr_id: parts.append(f'SR Number: {sr_id}')
    if title: parts.append(f'Title: {title}')
    if subject and subject != title: parts.append(f'Subject: {subject}')
    if description: parts.append(f'Description:\n{description}')
    if history: parts.append(f'SR History:\n{history}')
    content='\n\n'.join(parts).strip()
    if not content: return None
    metadata={'sr_id':sr_id,'title':title,'subject':subject,'source':source,'sr_details_url':source,'captured_at':clean_text(data.get('captured_at')),'content_type':'service_request','platform':'Oracle Service Request','source_file':path.name}
    return Document(page_content=content, metadata={k:v for k,v in metadata.items() if v})


In [ ]:
# CELL 3: Load all 1361 JSON files
files=sorted(SR_JSON_DIR.glob('*.json'))
print('JSON files found:',len(files))
documents=[]
failures=[]
for i,path in enumerate(files,1):
    try:
        data=json.loads(path.read_text(encoding='utf-8'))
        doc=build_document(data,path)
        if doc is None: raise ValueError('No searchable content')
        documents.append(doc)
        if i<=10 or i%100==0 or i==len(files): print(f"[{i}/{len(files)}] {doc.metadata.get('sr_id',path.stem)} - {doc.metadata.get('title','')}")
    except Exception as exc:
        failures.append({'file':path.name,'error':str(exc)})
        print(f'[{i}/{len(files)}] FAILED {path.name}: {exc}')
print('Loaded SR documents:',len(documents))
print('Failures:',len(failures))


In [ ]:
# CELL 4: Inspect sample
sample=documents[0]
print('SR:',sample.metadata.get('sr_id'))
print('Title:',sample.metadata.get('title'))
print('Source:',sample.metadata.get('source'))
print(sample.page_content[:3000])


In [ ]:
# CELL 5: Chunk
splitter=RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE,chunk_overlap=CHUNK_OVERLAP,separators=['\n\n','\n','. ',' ',''])
chunks=splitter.split_documents(documents)
print('Source SRs:',len(documents))
print('Chunks:',len(chunks))


In [ ]:
# CELL 6: Build ONLY SR Chroma DB
if not chunks: raise RuntimeError('No SR chunks created')
if CHROMA_DIR.exists(): shutil.rmtree(CHROMA_DIR)
embedding=HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL,model_kwargs={'device':'cpu'},encode_kwargs={'normalize_embeddings':True})
vector_db=Chroma.from_documents(chunks,embedding=embedding,persist_directory=str(CHROMA_DIR),collection_name=COLLECTION_NAME)
print('Collection:',COLLECTION_NAME)
print('Indexed chunks:',vector_db._collection.count())
Path('sr_ingestion_failures.json').write_text(json.dumps(failures,indent=2),encoding='utf-8')


In [ ]:
# CELL 7: Retrieval test
query='ACFS usage during GI patching on Exadata'
results=vector_db.similarity_search_with_score(query,k=5)
for i,(doc,score) in enumerate(results,1):
    print(f'\n--- RESULT {i} score={score} ---')
    print('SR:',doc.metadata.get('sr_id'))
    print('Title:',doc.metadata.get('title'))
    print('Source:',doc.metadata.get('source'))
    print('Text:',doc.page_content[:1000])
